In [ ]:
# In[1]:

import pandas as pd

# Load CSVs into DataFrames and parse timestamp as UTC datetimes
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# Convert timestamp columns to UTC datetimes (Unix seconds)
for df in (metric_df, trace_df, log_df, error_logs_df):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Helper to get top-N unique values (by occurrence) as list
def top_unique(series, n=20):
    if series is None or series.shape[0] == 0:
        return []
    return series.value_counts().index[:n].tolist()

# Summaries for each file
summaries = []

# metric.csv summary
summaries.append({
    "file": "metric.csv",
    "rows": metric_df.shape[0],
    "min_timestamp": metric_df['timestamp'].min(),
    "max_timestamp": metric_df['timestamp'].max(),
    "top_cmdb_ids": top_unique(metric_df['cmdb_id'], 20),
    "name_field": "kpi_name",
    "top_names": top_unique(metric_df['kpi_name'], 20)
})

# trace.csv summary
summaries.append({
    "file": "trace.csv",
    "rows": trace_df.shape[0],
    "min_timestamp": trace_df['timestamp'].min(),
    "max_timestamp": trace_df['timestamp'].max(),
    "top_cmdb_ids": top_unique(trace_df['cmdb_id'], 20),
    "name_field": "trace_name",
    "top_names": top_unique(trace_df['trace_name'], 20)
})

# log.csv summary
summaries.append({
    "file": "log.csv",
    "rows": log_df.shape[0],
    "min_timestamp": log_df['timestamp'].min(),
    "max_timestamp": log_df['timestamp'].max(),
    "top_cmdb_ids": top_unique(log_df['cmdb_id'], 20),
    "name_field": "log_name",
    "top_names": top_unique(log_df['log_name'], 20)
})

# error_logs.csv summary (messages handled separately)
summaries.append({
    "file": "error_logs.csv",
    "rows": error_logs_df.shape[0],
    "min_timestamp": error_logs_df['timestamp'].min(),
    "max_timestamp": error_logs_df['timestamp'].max(),
    "top_cmdb_ids": top_unique(error_logs_df['cmdb_id'], 20),
    "name_field": "message",
    "top_names": top_unique(error_logs_df['message'], 20)
})

# Create a compact DataFrame for summaries
summary_df = pd.DataFrame(summaries)[[
    "file", "rows", "min_timestamp", "max_timestamp", "top_cmdb_ids", "name_field", "top_names"
]]

# For error_logs.csv, compute top 5 distinct messages and counts
error_top5 = (error_logs_df['message']
              .value_counts()
              .head(5)
              .rename_axis('message')
              .reset_index(name='count'))

# Display compact outputs (store full DFs in variables for later use)
summary_df, error_top5

```
Out[1]:
```
```python
# Compose a concise plain-English summary using previously loaded variables
metric_row = summary_df[summary_df.file == "metric.csv"].iloc[0]
trace_row = summary_df[summary_df.file == "trace.csv"].iloc[0]
log_row = summary_df[summary_df.file == "log.csv"].iloc[0]
error_row = summary_df[summary_df.file == "error_logs.csv"].iloc[0]

def fmt_list(lst, n=6):
    return ", ".join(map(str, (lst[:n] if isinstance(lst, (list, tuple)) else list(lst)[:n])))

summary = (
    f"Summary of loaded telemetry files:\n\n"
    f"1) metric.csv: {metric_row.rows} rows; time range {metric_row.min_timestamp.isoformat()} to {metric_row.max_timestamp.isoformat()}. "
    f"Top CMDB ids (sample): {fmt_list(metric_row.top_cmdb_ids)}. Top KPI names (sample): {fmt_list(metric_row.top_names, 10)}.\n\n"
    f"2) trace.csv: {trace_row.rows} rows; time range {trace_row.min_timestamp.isoformat()} to {trace_row.max_timestamp.isoformat()}. "
    f"Top CMDB ids (sample): {fmt_list(trace_row.top_cmdb_ids)}. Top trace names (sample): {fmt_list(trace_row.top_names, 10)}.\n\n"
    f"3) log.csv: {log_row.rows} rows; time range {log_row.min_timestamp.isoformat()} to {log_row.max_timestamp.isoformat()}. "
    f"Top CMDB ids (sample): {fmt_list(log_row.top_cmdb_ids)}. Log name fields: {fmt_list(log_row.top_names, 10)}.\n\n"
    f"4) error_logs.csv: {error_row.rows} rows (empty). No timestamps or messages available.\n\n"
    f"Notes: All timestamps are in UTC. Full DataFrames are kept in metric_df, trace_df, log_df, and error_logs_df for further analysis."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(             file  rows             min_timestamp             max_timestamp                                       top_cmdb_ids  name_field                                          top_names
0      metric.csv  1750 2024-01-17 03:56:00+00:00 2024-01-17 04:20:00+00:00  [adservice, currencyservice, recommendationser...    kpi_name  [cpu, workload, socket, mem, latency-50, laten...
1       trace.csv  3800 2024-01-17 03:56:00+00:00 2024-01-17 04:20:00+00:00  [checkoutservice, frontendservice, root, recom...  trace_name  [trace.from_root.duration_mean, trace.from_roo...
2         log.csv   460 2024-01-17 03:56:00+00:00 2024-01-17 04:20:00+00:00  [adservice, cartservice, checkoutservice, curr...    log_name                   [log.error_count, log.row_count]
3  error_logs.csv     0                       NaT                       NaT                                                 []     message                                                 [], Empty DataFrame
Columns: [message, count]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd

# reuse existing DataFrames: metric_df, trace_df, log_df (assumed present in kernel)
# define incident window in UTC
start = pd.to_datetime("2024-01-17 03:53:39", utc=True)
end = pd.to_datetime("2024-01-17 04:23:39", utc=True)

def aggregate_alerts(df, name_field):
    # 1) compute global stats using entire series (before filtering)
    grp = df.groupby(['cmdb_id', name_field])['value']
    global_stats = grp.agg(
        count='size',
        global_P95=lambda s: s.quantile(0.95),
        global_P5=lambda s: s.quantile(0.05),
        median='median'
    ).reset_index()
    
    # 2) filter to incident window
    df_win = df[(df['timestamp'] >= start) & (df['timestamp'] <= end)].copy()
    
    # basic window aggregations
    win_agg = df_win.groupby(['cmdb_id', name_field])['value'].agg(
        count_in_window='size',
        max_value_in_window='max'
    ).reset_index()
    
    # join global thresholds into window rows to compute above-P95 flags per-row
    if df_win.shape[0] > 0:
        df_win = df_win.merge(global_stats[['cmdb_id', name_field, 'global_P95']], on=['cmdb_id', name_field], how='left')
        df_win['above_p95'] = df_win['value'] > df_win['global_P95']
        # per-group compute count_above_p95 and earliest timestamp above p95
        above_agg = df_win[df_win['above_p95']].groupby(['cmdb_id', name_field]).agg(
            count_above_p95=('above_p95', 'sum'),
            earliest_timestamp_above_p95=('timestamp', 'min')
        ).reset_index()
    else:
        # empty window: create empty above_agg with correct columns
        above_agg = pd.DataFrame(columns=['cmdb_id', name_field, 'count_above_p95', 'earliest_timestamp_above_p95'])
    
    # merge everything
    agg = global_stats.merge(win_agg, on=['cmdb_id', name_field], how='left')
    agg = agg.merge(above_agg, on=['cmdb_id', name_field], how='left')
    
    # fill NaNs for window-derived numeric fields with 0
    agg['count_in_window'] = agg['count_in_window'].fillna(0).astype(int)
    agg['count_above_p95'] = agg['count_above_p95'].fillna(0).astype(int)
    # earliest timestamp keep as NaT if missing
    if 'earliest_timestamp_above_p95' in agg.columns:
        agg['earliest_timestamp_above_p95'] = pd.to_datetime(agg['earliest_timestamp_above_p95'], utc=True)
    else:
        agg['earliest_timestamp_above_p95'] = pd.NaT
    
    # format earliest timestamp as ISO UTC strings, keep NaT as None
    agg['earliest_timestamp_above_p95'] = agg['earliest_timestamp_above_p95'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    
    # ensure column order and names: unify to requested names
    col_name = name_field
    result = agg[[
        'cmdb_id',
        col_name,
        'count',
        'count_in_window',
        'count_above_p95',
        'earliest_timestamp_above_p95',
        'global_P95',
        'global_P5',
        'median',
        'max_value_in_window'
    ]].copy()
    
    # sort by count_above_p95 desc, then count_in_window desc, then count desc
    result = result.sort_values(by=['count_above_p95', 'count_in_window', 'count'], ascending=[False, False, False])
    
    return result

# Compute for metric, trace, and log
metric_agg_df = aggregate_alerts(metric_df, 'kpi_name')
trace_agg_df = aggregate_alerts(trace_df, 'trace_name')
log_agg_df = aggregate_alerts(log_df, 'log_name')

# Keep only top 20 rows for display as required
metric_agg_df_head = metric_agg_df.head(20)
trace_agg_df_head = trace_agg_df.head(20)
log_agg_df_head = log_agg_df.head(20)

# Return the three compact tables (variables will be displayed by the IPython kernel)
metric_agg_df_head, trace_agg_df_head, log_agg_df_head

```
Out[2]:
```
```python
summary = (
    "Summary of findings for incident window 2024-01-17 03:53:39 to 2024-01-17 04:23:39 UTC:\n\n"
    "- Multiple services (notably adservice, cartservice, checkoutservice, currencyservice and others) "
    "had metric values that exceeded their global 95th percentile during the window. Key metric types with spikes: "
    "cpu, mem, workload, disk I/O, and latency (latency-50 and latency-90). Each affected (cmdb_id, metric) pair shows "
    "a small number of spikes (count_above_p95 = 2) within the 25 samples in the window.\n\n"
    "- Trace telemetry shows concentrated anomalies around checkoutservice (many from_*/to_* and root traces) with "
    "trace latencies and row_count exceeding global P95 (also typically 2 points above P95). Earliest exceedances "
    "occurred roughly between ~03:58 and ~04:18 UTC (examples in results: adservice.cpu 04:01, adservice.diskio 04:03, "
    "checkoutservice traces around 04:14–04:18 UTC).\n\n"
    "- Log telemetry: row_count increased above P95 for many services (count_above_p95 = 2), while log.error_count "
    "did not spike (no error_count exceedances). There are no error_log entries.\n\n"
    "Interpretation: the incident appears to be short-lived performance/resource spikes (CPU, memory, workload, I/O, and "
    "latency) across multiple services, with checkoutservice trace latencies being particularly prominent. There is no "
    "clear spike in error logs. Recommended next steps: investigate resource usage and request/response latencies for "
    "checkoutservice and its dependencies (e.g., currencyservice, paymentservice) around ~04:00–04:18 UTC."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id    kpi_name  count  count_in_window  count_above_p95 earliest_timestamp_above_p95    global_P95     global_P5        median  max_value_in_window
0         adservice         cpu     25               25                2         2024-01-17T04:01:00Z  7.619850e-01  5.603580e-01  6.626969e-01         7.993782e-01
1         adservice      diskio     25               25                2         2024-01-17T04:03:00Z  7.727903e+04  5.207772e+04  6.196789e+04         8.984126e+04
2         adservice  latency-50     25               25                2         2024-01-17T04:10:00Z  3.064242e-03  2.995601e-03  3.009122e-03         3.089878e-03
3         adservice  latency-90     25               25                2         2024-01-17T04:10:00Z  4.729194e-03  4.606496e-03  4.641288e-03         4.776578e-03
4         adservice         mem     25               25                2         2024-01-17T04:16:00Z  1.073137e+08  1.064805e+08  1.068764e+08         1.073572e+08
6         adservice    workload     25               25                2         2024-01-17T03:58:00Z  1.546261e+01  1.270004e+01  1.481005e+01         1.561345e+01
7       cartservice         cpu     25               25                2         2024-01-17T03:58:00Z  2.586709e+00  1.390620e+00  1.907703e+00         2.704271e+00
8       cartservice  latency-50     25               25                2         2024-01-17T04:14:00Z  5.474352e-03  3.309506e-03  3.915928e-03         5.622108e-03
9       cartservice  latency-90     25               25                2         2024-01-17T04:15:00Z  1.559900e-02  6.312960e-03  8.892377e-03         1.749563e-02
10      cartservice         mem     25               25                2         2024-01-17T04:12:00Z  4.372533e+07  4.345048e+07  4.353399e+07         4.373900e+07
12      cartservice    workload     25               25                2         2024-01-17T03:58:00Z  2.625688e+01  2.109161e+01  2.332043e+01         2.693120e+01
13  checkoutservice         cpu     25               25                2         2024-01-17T03:58:00Z  4.649672e-01  3.134284e-01  3.873420e-01         4.740316e-01
14  checkoutservice  latency-50     25               25                2         2024-01-17T04:18:00Z  7.344464e-01  1.227186e-01  1.575298e-01         8.312864e-01
15  checkoutservice  latency-90     25               25                2         2024-01-17T04:18:00Z  2.306130e+00  2.289584e-01  3.104137e-01         2.665540e+00
16  checkoutservice         mem     25               25                2         2024-01-17T04:00:00Z  1.184507e+07  1.149226e+07  1.166766e+07         1.186150e+07
18  checkoutservice    workload     25               25                2         2024-01-17T03:58:00Z  1.050717e+00  5.489052e-01  8.632667e-01         1.255550e+00
19  currencyservice         cpu     25               25                2         2024-01-17T04:11:00Z  3.958943e+01  2.701862e+01  3.065501e+01         3.997479e+01
21  currencyservice  latency-50     25               25                2         2024-01-17T04:16:00Z  1.043986e-01  4.148017e-03  4.543025e-03         1.099837e-01
22  currencyservice  latency-90     25               25                2         2024-01-17T04:10:00Z  7.544970e-01  6.626651e-02  7.378074e-02         7.910584e-01
23  currencyservice         mem     25               25                2         2024-01-17T04:11:00Z  2.516954e+08  4.454885e+07  4.857501e+07         2.614594e+08,             cmdb_id                                trace_name  count  count_in_window  count_above_p95 earliest_timestamp_above_p95  global_P95   global_P5      median  max_value_in_window
0   checkoutservice  trace.from_checkoutservice.duration_mean     25               25                2         2024-01-17T04:14:00Z    0.015334    0.003483    0.003836             0.016410
1   checkoutservice   trace.from_checkoutservice.duration_p95     25               25                2         2024-01-17T04:18:00Z    0.008538    0.005313    0.006014             0.042266
3   checkoutservice      trace.from_checkoutservice.row_count     25               25                2         2024-01-17T03:58:00Z  282.200000  138.600000  222.000000           322.000000
4   checkoutservice  trace.from_frontendservice.duration_mean     25               25                2         2024-01-17T04:18:00Z    0.959058    0.111256    0.364129             1.419229
5   checkoutservice   trace.from_frontendservice.duration_p95     25               25                2         2024-01-17T04:15:00Z    2.258047    0.204401    1.468737             3.416767
7   checkoutservice      trace.from_frontendservice.row_count     25               25                2         2024-01-17T03:58:00Z   34.800000   16.200000   26.000000            37.000000
8   checkoutservice             trace.from_root.duration_mean     25               25                2         2024-01-17T04:18:00Z    0.183627    0.018228    0.077618             0.254955
9   checkoutservice              trace.from_root.duration_p95     25               25                2         2024-01-17T04:16:00Z    1.104169    0.074918    0.189743             1.184442
11  checkoutservice                 trace.from_root.row_count     25               25                2         2024-01-17T03:58:00Z  132.200000   79.800000  111.000000           157.000000
12  checkoutservice    trace.to_checkoutservice.duration_mean     25               25                2         2024-01-17T04:14:00Z    0.015334    0.003483    0.003836             0.016410
13  checkoutservice     trace.to_checkoutservice.duration_p95     25               25                2         2024-01-17T04:18:00Z    0.008538    0.005313    0.006014             0.042266
15  checkoutservice        trace.to_checkoutservice.row_count     25               25                2         2024-01-17T03:58:00Z  282.200000  138.600000  222.000000           322.000000
16  checkoutservice    trace.to_currencyservice.duration_mean     25               25                2         2024-01-17T04:13:00Z    0.006055    0.000112    0.000154             0.008705
17  checkoutservice     trace.to_currencyservice.duration_p95     25               25                2         2024-01-17T04:16:00Z    0.000255    0.000193    0.000214             0.001166
19  checkoutservice        trace.to_currencyservice.row_count     25               25                2         2024-01-17T03:58:00Z  108.200000   57.000000   87.000000           133.000000
20  checkoutservice       trace.to_emailservice.duration_mean     25               25                2         2024-01-17T03:57:00Z    0.000372    0.000258    0.000315             0.000384
21  checkoutservice        trace.to_emailservice.duration_p95     25               25                2         2024-01-17T04:08:00Z    0.000786    0.000333    0.000500             0.000874
23  checkoutservice           trace.to_emailservice.row_count     25               25                2         2024-01-17T03:58:00Z   34.800000   16.200000   26.000000            38.000000
24  checkoutservice     trace.to_paymentservice.duration_mean     25               25                2         2024-01-17T04:15:00Z    0.000311    0.000203    0.000247             0.000334
25  checkoutservice      trace.to_paymentservice.duration_p95     25               25                2         2024-01-17T04:18:00Z    0.000612    0.000290    0.000390             0.001030,                   cmdb_id         log_name  count  count_in_window  count_above_p95 earliest_timestamp_above_p95  global_P95  global_P5  median  max_value_in_window
1               adservice    log.row_count     25               25                2         2024-01-17T04:03:00Z       461.6      278.2   399.0                  473
3             cartservice    log.row_count     25               25                2         2024-01-17T03:58:00Z       796.4      481.6   689.0                  813
5         checkoutservice    log.row_count     25               25                2         2024-01-17T03:58:00Z       104.4       48.6    78.0                  113
7         currencyservice    log.row_count     25               25                2         2024-01-17T03:58:00Z      2492.6     1441.6  2168.0                 2523
9            emailservice    log.row_count     25               25                2         2024-01-17T03:58:00Z        34.8       16.2    26.0                   38
11               frontend    log.row_count     25               25                2         2024-01-17T03:58:00Z      2772.4     1658.0  2403.0                 2818
13         paymentservice    log.row_count     25               25                2         2024-01-17T03:58:00Z        69.6       32.4    52.0                   76
15  recommendationservice    log.row_count     25               25                2         2024-01-17T04:03:00Z       577.6      349.0   502.0                  587
19        shippingservice    log.row_count     25               25                2         2024-01-17T03:58:00Z       464.4      276.0   410.0                  476
0               adservice  log.error_count     25               25                0                          NaN         0.0        0.0     0.0                    0
2             cartservice  log.error_count     25               25                0                          NaN         0.0        0.0     0.0                    0
4         checkoutservice  log.error_count     25               25                0                          NaN         0.0        0.0     0.0                    0
6         currencyservice  log.error_count     25               25                0                          NaN         0.0        0.0     0.0                    0
8            emailservice  log.error_count     25               25                0                          NaN         0.0        0.0     0.0                    0
10               frontend  log.error_count     25               25                0                          NaN         0.0        0.0     0.0                    0
12         paymentservice  log.error_count     25               25                0                          NaN         0.0        0.0     0.0                    0
14  recommendationservice  log.error_count     25               25                0                          NaN         0.0        0.0     0.0                    0
18        shippingservice  log.error_count     25               25                0                          NaN         0.0        0.0     0.0                    0
16                  redis  log.error_count      5                5                0                          NaN         0.0        0.0     0.0                    0
17                  redis    log.row_count      5                5                0                          NaN         5.0        5.0     5.0                    5)```
```

In [ ]:
# In[3]:

import pandas as pd

# reuse existing metric_df and metric_agg_df from kernel
start = pd.to_datetime("2024-01-17 03:53:39", utc=True)
end = pd.to_datetime("2024-01-17 04:23:39", utc=True)

rows = []
# iterate only pairs that had exceedances
candidates = metric_agg_df[metric_agg_df['count_above_p95'] > 0]

for _, r in candidates.iterrows():
    cmdb = r['cmdb_id']
    kpi = r['kpi_name']
    global_p95 = r['global_P95']
    # filter metric_df to this pair and incident window
    s = metric_df[
        (metric_df['cmdb_id'] == cmdb) &
        (metric_df['kpi_name'] == kpi) &
        (metric_df['timestamp'] >= start) &
        (metric_df['timestamp'] <= end)
    ].copy()
    if s.empty:
        continue
    # keep only points above global P95
    s = s[s['value'] > global_p95].sort_values('timestamp')
    if s.empty:
        continue
    # identify consecutive runs (timestamps spaced by exactly 1 minute)
    diff = s['timestamp'].diff()
    new_run = (diff != pd.Timedelta(minutes=1))
    # first row should start a run
    new_run.iloc[0] = True
    run_id = new_run.cumsum()
    s['run_id'] = run_id.values
    # aggregate runs
    grp = s.groupby('run_id').agg(
        start_timestamp=('timestamp', 'min'),
        end_timestamp=('timestamp', 'max'),
        length=('timestamp', 'size'),
        max_value_in_run=('value', 'max')
    ).reset_index(drop=True)
    # keep runs with length >= 2
    grp = grp[grp['length'] >= 2]
    if grp.empty:
        continue
    # append rows
    for _, g in grp.iterrows():
        rows.append({
            'cmdb_id': cmdb,
            'kpi_name': kpi,
            'start_timestamp': g['start_timestamp'],
            'end_timestamp': g['end_timestamp'],
            'length': int(g['length']),
            'max_value_in_run': g['max_value_in_run'],
            'global_P95': global_p95
        })

# full results stored in kernel variable
metric_faults_df = pd.DataFrame(rows)

# format timestamps as ISO UTC strings and ensure column order
if not metric_faults_df.empty:
    metric_faults_df['start_timestamp'] = pd.to_datetime(metric_faults_df['start_timestamp'], utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    metric_faults_df['end_timestamp'] = pd.to_datetime(metric_faults_df['end_timestamp'], utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    metric_faults_df = metric_faults_df[[
        'cmdb_id', 'kpi_name', 'start_timestamp', 'end_timestamp', 'length', 'max_value_in_run', 'global_P95'
    ]]
else:
    # ensure consistent columns even if empty
    metric_faults_df = pd.DataFrame(columns=[
        'cmdb_id', 'kpi_name', 'start_timestamp', 'end_timestamp', 'length', 'max_value_in_run', 'global_P95'
    ])

# return compact top 20 runs sorted by length desc then max_value_in_run desc
metric_faults_df = metric_faults_df.sort_values(by=['length', 'max_value_in_run'], ascending=[False, False]).reset_index(drop=True)
metric_faults_df_head = metric_faults_df.head(20)

metric_faults_df_head

```
Out[3]:
```
Summary of consecutive anomaly runs detected within the incident window (2024-01-17 03:53:39–04:23:39 UTC):

- We found multiple short consecutive anomaly runs (length = 2 minutes) where metric values exceeded their global P95.
- Affected metrics and services (top runs):
  - adservice — mem: 2024-01-17T04:16:00Z to 04:17:00Z (2 points).
  - recommendationservice — mem: 04:17:00Z to 04:18:00Z (2 points).
  - emailservice — mem: 04:19:00Z to 04:20:00Z (2 points).
  - paymentservice — mem: 04:18:00Z to 04:19:00Z (2 points).
  - productcatalogservice — mem: 04:14:00Z to 04:15:00Z (2 points) and cpu: 04:04:00Z to 04:05:00Z (2 points).
  - checkoutservice — latency-90 and latency-50: 04:18:00Z to 04:19:00Z (both 2 points).
  - shippingservice — latency-90: 04:18:00Z to 04:19:00Z (2 points).
  - productcatalogservice — latency-90: 04:19:00Z to 04:20:00Z (2 points).

Interpretation and next steps:
- The main pattern is short-lived (2-minute) spikes in memory across several services and simultaneous latency spikes (notably checkoutservice and shippingservice) around 04:14–04:20 UTC.
- Recommend investigating memory pressure and recent load changes across the listed services, and tracing request flows for checkoutservice/shippingservice during ~04:18 UTC to find dependency or resource bottlenecks.
- Full run records are stored in the kernel variable metric_faults_df for follow-up.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name       start_timestamp         end_timestamp  length  max_value_in_run    global_P95
0              adservice         mem  2024-01-17T04:16:00Z  2024-01-17T04:17:00Z       2      1.073572e+08  1.073137e+08
1  recommendationservice         mem  2024-01-17T04:17:00Z  2024-01-17T04:18:00Z       2      4.543863e+07  4.543300e+07
2           emailservice         mem  2024-01-17T04:19:00Z  2024-01-17T04:20:00Z       2      4.339763e+07  4.339354e+07
3         paymentservice         mem  2024-01-17T04:18:00Z  2024-01-17T04:19:00Z       2      3.886981e+07  3.871762e+07
4  productcatalogservice         mem  2024-01-17T04:14:00Z  2024-01-17T04:15:00Z       2      1.069582e+07  1.055078e+07
5  productcatalogservice         cpu  2024-01-17T04:04:00Z  2024-01-17T04:05:00Z       2      3.763773e+00  3.732582e+00
6        checkoutservice  latency-90  2024-01-17T04:18:00Z  2024-01-17T04:19:00Z       2      2.665540e+00  2.306130e+00
7        checkoutservice  latency-50  2024-01-17T04:18:00Z  2024-01-17T04:19:00Z       2      8.312864e-01  7.344464e-01
8        shippingservice  latency-90  2024-01-17T04:18:00Z  2024-01-17T04:19:00Z       2      4.648777e-03  4.633023e-03
9  productcatalogservice  latency-90  2024-01-17T04:19:00Z  2024-01-17T04:20:00Z       2      4.640537e-03  4.607207e-03```
```

In [ ]:
# In[4]:

import pandas as pd

# reuse existing trace_agg_df from kernel
# 1) Top trace anomalies: count_above_p95 > 0
trace_anom_top_df = trace_agg_df[trace_agg_df['count_above_p95'] > 0].copy()

# normalize earliest timestamp column to datetime for sorting, keeping ISO strings for output
trace_anom_top_df['_earliest_dt'] = pd.to_datetime(trace_anom_top_df['earliest_timestamp_above_p95'], utc=True, errors='coerce')
trace_anom_top_df = trace_anom_top_df.sort_values(by='_earliest_dt', ascending=True).reset_index(drop=True)

# select requested columns and ensure earliest_timestamp is ISO UTC string
cols = ['cmdb_id', 'trace_name', 'count', 'count_in_window', 'count_above_p95',
        'earliest_timestamp_above_p95', 'global_P95', 'max_value_in_window']
trace_anom_top_df = trace_anom_top_df[cols]

# keep a display-limited head (top 20) but store full filtered in variable as required
trace_anom_top_df_head = trace_anom_top_df.head(20)

# 2) Checkoutservice traces with anomalies
trace_checkout_df = trace_agg_df[
    (trace_agg_df['cmdb_id'] == 'checkoutservice') & (trace_agg_df['count_above_p95'] > 0)
].copy()

trace_checkout_df['_earliest_dt'] = pd.to_datetime(trace_checkout_df['earliest_timestamp_above_p95'], utc=True, errors='coerce')
trace_checkout_df = trace_checkout_df.sort_values(by='_earliest_dt', ascending=True).reset_index(drop=True)
trace_checkout_df = trace_checkout_df[cols]

trace_checkout_df_head = trace_checkout_df.head(20)

# Return the two compact tables (head views); full filtered DataFrames are stored in
# trace_anom_top_df and trace_checkout_df per requirement.
trace_anom_top_df_head, trace_checkout_df_head

```
Out[4]:
```
```python
summary = (
    "Summary of trace anomalies (plain English):\n\n"
    "- Overall: Multiple trace metrics exceeded their global P95 during the incident window, "
    "with the earliest anomalies appearing around 2024-01-17T03:56:00Z and a concentrated set of "
    "row_count spikes at ~2024-01-17T03:58:00Z.\n\n"
    "- Top anomalies (examples): duration p95/mean spikes involving recommendationservice, "
    "productcatalogservice, root spans, and frontend-related row_count spikes starting ~03:56–03:58 UTC.\n\n"
    "- Checkoutservice: Many checkoutservice traces exceeded P95 (duration and row_count). "
    "Earliest checkoutservice anomalies begin as early as 2024-01-17T03:56:00Z (duration.p95) and "
    "include row_count spikes at ~03:58:00Z and latency/duration spikes around 04:14–04:18Z.\n\n"
    "- Interpretation and next steps: Checkoutservice shows prominent latency and traffic anomalies and "
    "is a likely focal point. Investigate request rates and latencies at checkoutservice and its downstream "
    "dependencies (productcatalogservice, paymentservice, emailservice, currencyservice, frontend) around "
    "~03:56–04:18 UTC to find root cause and correlated resource issues."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id                                     trace_name  count  count_in_window  count_above_p95 earliest_timestamp_above_p95   global_P95  max_value_in_window
0   recommendationservice    trace.to_productcatalogservice.duration_p95     25               25                2         2024-01-17T03:56:00Z     0.000034             0.000038
1   productcatalogservice        trace.from_checkoutservice.duration_p95     25               25                2         2024-01-17T03:56:00Z     0.000039             0.000085
2   productcatalogservice  trace.from_recommendationservice.duration_p95     25               25                2         2024-01-17T03:56:00Z     0.000034             0.000038
3                    root   trace.to_recommendationservice.duration_mean     25               25                2         2024-01-17T03:56:00Z     0.000054             0.000059
4         checkoutservice    trace.to_productcatalogservice.duration_p95     25               25                2         2024-01-17T03:56:00Z     0.000039             0.000085
5   recommendationservice                  trace.from_root.duration_mean     25               25                2         2024-01-17T03:56:00Z     0.000054             0.000059
6         checkoutservice            trace.to_emailservice.duration_mean     25               25                2         2024-01-17T03:57:00Z     0.000372             0.000384
7            emailservice       trace.from_checkoutservice.duration_mean     25               25                2         2024-01-17T03:57:00Z     0.000372             0.000384
8                    root             trace.to_frontendservice.row_count     25               25                2         2024-01-17T03:58:00Z   841.800000           861.000000
9         frontendservice             trace.to_frontendservice.row_count     25               25                2         2024-01-17T03:58:00Z  8023.800000          8232.000000
10        frontendservice             trace.to_currencyservice.row_count     25               25                2         2024-01-17T03:58:00Z  2385.800000          2406.000000
11        frontendservice             trace.to_checkoutservice.row_count     25               25                2         2024-01-17T03:58:00Z    34.800000            37.000000
12        currencyservice           trace.from_frontendservice.row_count     25               25                2         2024-01-17T03:58:00Z  2385.800000          2406.000000
13        frontendservice                      trace.from_root.row_count     25               25                2         2024-01-17T03:58:00Z   841.800000           861.000000
14        checkoutservice              trace.to_paymentservice.row_count     25               25                2         2024-01-17T03:58:00Z    34.800000            38.000000
15        frontendservice           trace.from_frontendservice.row_count     25               25                2         2024-01-17T03:58:00Z  8023.800000          8232.000000
16        currencyservice           trace.from_checkoutservice.row_count     25               25                2         2024-01-17T03:58:00Z   108.200000           133.000000
17                   root          trace.to_paymentservice.duration_mean     25               25                2         2024-01-17T03:58:00Z     0.000072             0.000072
18           emailservice           trace.from_checkoutservice.row_count     25               25                2         2024-01-17T03:58:00Z    34.800000            38.000000
19                   root             trace.to_checkoutservice.row_count     25               25                2         2024-01-17T03:58:00Z   132.200000           157.000000,             cmdb_id                                    trace_name  count  count_in_window  count_above_p95 earliest_timestamp_above_p95  global_P95  max_value_in_window
0   checkoutservice   trace.to_productcatalogservice.duration_p95     25               25                2         2024-01-17T03:56:00Z    0.000039             0.000085
1   checkoutservice           trace.to_emailservice.duration_mean     25               25                2         2024-01-17T03:57:00Z    0.000372             0.000384
2   checkoutservice            trace.to_checkoutservice.row_count     25               25                2         2024-01-17T03:58:00Z  282.200000           322.000000
3   checkoutservice             trace.to_paymentservice.row_count     25               25                2         2024-01-17T03:58:00Z   34.800000            38.000000
4   checkoutservice               trace.to_emailservice.row_count     25               25                2         2024-01-17T03:58:00Z   34.800000            38.000000
5   checkoutservice            trace.to_currencyservice.row_count     25               25                2         2024-01-17T03:58:00Z  108.200000           133.000000
6   checkoutservice                     trace.from_root.row_count     25               25                2         2024-01-17T03:58:00Z  132.200000           157.000000
7   checkoutservice      trace.to_productcatalogservice.row_count     25               25                1         2024-01-17T03:58:00Z   75.000000            95.000000
8   checkoutservice          trace.from_frontendservice.row_count     25               25                2         2024-01-17T03:58:00Z   34.800000            37.000000
9   checkoutservice          trace.from_checkoutservice.row_count     25               25                2         2024-01-17T03:58:00Z  282.200000           322.000000
10  checkoutservice  trace.to_productcatalogservice.duration_mean     25               25                2         2024-01-17T04:02:00Z    0.000027             0.000029
11  checkoutservice            trace.to_emailservice.duration_p95     25               25                2         2024-01-17T04:08:00Z    0.000786             0.000874
12  checkoutservice        trace.to_currencyservice.duration_mean     25               25                2         2024-01-17T04:13:00Z    0.006055             0.008705
13  checkoutservice        trace.to_checkoutservice.duration_mean     25               25                2         2024-01-17T04:14:00Z    0.015334             0.016410
14  checkoutservice      trace.from_checkoutservice.duration_mean     25               25                2         2024-01-17T04:14:00Z    0.015334             0.016410
15  checkoutservice       trace.from_frontendservice.duration_p95     25               25                2         2024-01-17T04:15:00Z    2.258047             3.416767
16  checkoutservice         trace.to_paymentservice.duration_mean     25               25                2         2024-01-17T04:15:00Z    0.000311             0.000334
17  checkoutservice         trace.to_currencyservice.duration_p95     25               25                2         2024-01-17T04:16:00Z    0.000255             0.001166
18  checkoutservice                  trace.from_root.duration_p95     25               25                2         2024-01-17T04:16:00Z    1.104169             1.184442
19  checkoutservice                 trace.from_root.duration_mean     25               25                2         2024-01-17T04:18:00Z    0.183627             0.254955)```
```

In [ ]:
# In[5]:

import pandas as pd

# reuse existing variables from the kernel:
# metric_faults_df, trace_df, log_df (assumed present)

# incident window (UTC)
start = pd.to_datetime("2024-01-17 03:53:39", utc=True)
end = pd.to_datetime("2024-01-17 04:23:39", utc=True)

# 1) From metric_faults_df, filter for currencyservice or checkoutservice
if 'metric_faults_df' in globals():
    metric_currency_checkout_faults = metric_faults_df[
        metric_faults_df['cmdb_id'].isin(['currencyservice', 'checkoutservice'])
    ].copy()
else:
    metric_currency_checkout_faults = pd.DataFrame(columns=[
        'cmdb_id', 'kpi_name', 'start_timestamp', 'end_timestamp', 'length', 'max_value_in_run', 'global_P95'
    ])

# Ensure start/end timestamps are ISO UTC strings (they likely already are)
if not metric_currency_checkout_faults.empty:
    # If timestamps are datetimes, format; if strings, leave as-is
    for col in ['start_timestamp', 'end_timestamp']:
        if pd.api.types.is_datetime64_any_dtype(metric_currency_checkout_faults[col]):
            metric_currency_checkout_faults[col] = metric_currency_checkout_faults[col].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# compact view (top 20)
metric_currency_checkout_faults_head = metric_currency_checkout_faults.head(20)[
    ['cmdb_id', 'kpi_name', 'start_timestamp', 'end_timestamp', 'length', 'max_value_in_run', 'global_P95']
].reset_index(drop=True)

# 2) From trace_df, filter to incident window and the specified patterns
# Build boolean mask for the two conditions
mask_window = (trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)

mask_checkout_to_from_currency = (
    (trace_df['cmdb_id'] == 'checkoutservice') &
    trace_df['trace_name'].str.contains('to_currencyservice|from_currencyservice', regex=True)
)

mask_currency_to_from_checkout = (
    (trace_df['cmdb_id'] == 'currencyservice') &
    trace_df['trace_name'].str.contains('from_checkoutservice|to_checkoutservice', regex=True)
)

mask_traces = mask_window & (mask_checkout_to_from_currency | mask_currency_to_from_checkout)

traces_checkout_currency_window = trace_df[mask_traces].copy()

# Sort by timestamp ascending
if not traces_checkout_currency_window.empty:
    traces_checkout_currency_window = traces_checkout_currency_window.sort_values('timestamp').reset_index(drop=True)

# Prepare compact view with ISO timestamps and selected columns, limit 20
if not traces_checkout_currency_window.empty:
    traces_checkout_currency_window_head = traces_checkout_currency_window.loc[:, ['timestamp', 'cmdb_id', 'trace_name', 'value']].copy()
    traces_checkout_currency_window_head['timestamp'] = traces_checkout_currency_window_head['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    traces_checkout_currency_window_head = traces_checkout_currency_window_head.head(20).reset_index(drop=True)
else:
    traces_checkout_currency_window_head = pd.DataFrame(columns=['timestamp', 'cmdb_id', 'trace_name', 'value'])

# 3) From log_df, filter to incident window and cmdb_id in ['currencyservice','checkoutservice']
mask_log_window = (log_df['timestamp'] >= start) & (log_df['timestamp'] <= end)
mask_log_cmdb = log_df['cmdb_id'].isin(['currencyservice', 'checkoutservice'])
logs_checkout_currency_window = log_df[mask_log_window & mask_log_cmdb].copy()

# Sort ascending by timestamp
if not logs_checkout_currency_window.empty:
    logs_checkout_currency_window = logs_checkout_currency_window.sort_values('timestamp').reset_index(drop=True)

# Compact view with ISO timestamps and selected columns, limit 20
if not logs_checkout_currency_window.empty:
    logs_checkout_currency_window_head = logs_checkout_currency_window.loc[:, ['timestamp', 'cmdb_id', 'log_name', 'value']].copy()
    logs_checkout_currency_window_head['timestamp'] = logs_checkout_currency_window_head['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    logs_checkout_currency_window_head = logs_checkout_currency_window_head.head(20).reset_index(drop=True)
else:
    logs_checkout_currency_window_head = pd.DataFrame(columns=['timestamp', 'cmdb_id', 'log_name', 'value'])

# Return only the three compact tables (heads). Full filtered DataFrames are stored in:
# metric_currency_checkout_faults, traces_checkout_currency_window, logs_checkout_currency_window
metric_currency_checkout_faults_head, traces_checkout_currency_window_head, logs_checkout_currency_window_head

```
Out[5]:
```
Short answer: Currencyservice is suspicious but not proven to be the root cause.

Evidence:
- metric_faults_df: No currencyservice metric runs were detected as consecutive P95-exceeding faults. The metric faults shown are for checkoutservice (latency-50/90 at 2024-01-17T04:18–04:19) and several other services' mem/cpu — currencyservice does not appear in the metric faults table.
- trace evidence (03:56–03:58 UTC): Many traces show traffic between checkoutservice and currencyservice early in the window (checkout → currency and currency ← checkout). Row_count values are much higher on currencyservice (logs/traces show currencyservice row_count in the hundreds–thousands at 03:56–03:59) while checkoutservice row_count is an order of magnitude smaller. This indicates heavy traffic involving currencyservice early in the window.
- logs: currencyservice log.row_count spikes (e.g., ~853 at 03:56, ~2375 at 03:57, ~2523 at 03:58) whereas log.error_count stayed at 0. Checkoutservice row_count also rose but to much smaller values.

Interpretation and recommended next steps:
- The strong evidence is that currencyservice experienced a large increase in request volume early in the incident window, so it is a likely contributing factor (high load). However, because:
  - there are no consecutive metric-level faults recorded for currencyservice (no sustained P95-exceeding metric runs),
  - the prominent checkoutservice latency faults occur later (~04:18 UTC),
  we cannot conclusively attribute the root cause to currencyservice from the provided telemetry alone.
- To validate root cause, collect/inspect:
  - currencyservice resource metrics (CPU, memory, disk I/O) and host-level metrics around 03:56–04:00,
  - detailed traces linking the 03:56 traffic spike to downstream effects and the later 04:14–04:19 latency spikes,
  - any deployment/config changes or traffic pattern changes before 03:56.
- Conclusion: Currencyservice had abnormal high traffic and is a prime candidate to investigate further, but more metric/resource evidence is required to confirm it as the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(           cmdb_id    kpi_name       start_timestamp         end_timestamp  length  max_value_in_run  global_P95
0  checkoutservice  latency-90  2024-01-17T04:18:00Z  2024-01-17T04:19:00Z       2          2.665540    2.306130
1  checkoutservice  latency-50  2024-01-17T04:18:00Z  2024-01-17T04:19:00Z       2          0.831286    0.734446,                timestamp          cmdb_id                                trace_name       value
0   2024-01-17T03:56:00Z  checkoutservice    trace.to_currencyservice.duration_mean    0.000125
1   2024-01-17T03:56:00Z  checkoutservice     trace.to_currencyservice.duration_p95    0.000185
2   2024-01-17T03:56:00Z  checkoutservice       trace.to_currencyservice.error_rate    0.000000
3   2024-01-17T03:56:00Z  checkoutservice        trace.to_currencyservice.row_count   15.000000
4   2024-01-17T03:56:00Z  currencyservice  trace.from_checkoutservice.duration_mean    0.000125
5   2024-01-17T03:56:00Z  currencyservice   trace.from_checkoutservice.duration_p95    0.000185
6   2024-01-17T03:56:00Z  currencyservice     trace.from_checkoutservice.error_rate    0.000000
7   2024-01-17T03:56:00Z  currencyservice      trace.from_checkoutservice.row_count   15.000000
8   2024-01-17T03:57:00Z  currencyservice      trace.from_checkoutservice.row_count   70.000000
9   2024-01-17T03:57:00Z  currencyservice     trace.from_checkoutservice.error_rate    0.000000
10  2024-01-17T03:57:00Z  currencyservice  trace.from_checkoutservice.duration_mean    0.000146
11  2024-01-17T03:57:00Z  currencyservice   trace.from_checkoutservice.duration_p95    0.000216
12  2024-01-17T03:57:00Z  checkoutservice       trace.to_currencyservice.error_rate    0.000000
13  2024-01-17T03:57:00Z  checkoutservice     trace.to_currencyservice.duration_p95    0.000216
14  2024-01-17T03:57:00Z  checkoutservice    trace.to_currencyservice.duration_mean    0.000146
15  2024-01-17T03:57:00Z  checkoutservice        trace.to_currencyservice.row_count   70.000000
16  2024-01-17T03:58:00Z  checkoutservice    trace.to_currencyservice.duration_mean    0.001083
17  2024-01-17T03:58:00Z  checkoutservice     trace.to_currencyservice.duration_p95    0.000238
18  2024-01-17T03:58:00Z  checkoutservice       trace.to_currencyservice.error_rate    0.000000
19  2024-01-17T03:58:00Z  checkoutservice        trace.to_currencyservice.row_count  133.000000,                timestamp          cmdb_id         log_name  value
0   2024-01-17T03:56:00Z  checkoutservice  log.error_count      0
1   2024-01-17T03:56:00Z  checkoutservice    log.row_count     15
2   2024-01-17T03:56:00Z  currencyservice  log.error_count      0
3   2024-01-17T03:56:00Z  currencyservice    log.row_count    853
4   2024-01-17T03:57:00Z  checkoutservice  log.error_count      0
5   2024-01-17T03:57:00Z  checkoutservice    log.row_count     58
6   2024-01-17T03:57:00Z  currencyservice  log.error_count      0
7   2024-01-17T03:57:00Z  currencyservice    log.row_count   2375
8   2024-01-17T03:58:00Z  checkoutservice  log.error_count      0
9   2024-01-17T03:58:00Z  checkoutservice    log.row_count    113
10  2024-01-17T03:58:00Z  currencyservice  log.error_count      0
11  2024-01-17T03:58:00Z  currencyservice    log.row_count   2523
12  2024-01-17T03:59:00Z  currencyservice    log.row_count   2168
13  2024-01-17T03:59:00Z  currencyservice  log.error_count      0
14  2024-01-17T03:59:00Z  checkoutservice  log.error_count      0
15  2024-01-17T03:59:00Z  checkoutservice    log.row_count     78
16  2024-01-17T04:00:00Z  checkoutservice  log.error_count      0
17  2024-01-17T04:00:00Z  checkoutservice    log.row_count     72
18  2024-01-17T04:00:00Z  currencyservice  log.error_count      0
19  2024-01-17T04:00:00Z  currencyservice    log.row_count   2455)```
```

In [ ]:
# In[6]:

import pandas as pd
# reuse metric_df from kernel

# Targets and incident window (UTC)
targets = ['currencyservice', 'checkoutservice']
start = pd.to_datetime("2024-01-17 03:53:39", utc=True)
end = pd.to_datetime("2024-01-17 04:23:39", utc=True)

# 1) Compute global thresholds (P95 and P90) per (cmdb_id, kpi_name) using the full available series
mask_targets_full = metric_df['cmdb_id'].isin(targets)
metric_thresholds_currency_checkout = (
    metric_df[mask_targets_full]
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(
        count='size',
        global_P95=lambda s: s.quantile(0.95),
        global_P90=lambda s: s.quantile(0.90)
    )
    .reset_index()
)

# 2) Filter metric_df to the incident window and these services, merge thresholds, and find rows > global_P90
mask_window_targets = (
    (metric_df['timestamp'] >= start) &
    (metric_df['timestamp'] <= end) &
    metric_df['cmdb_id'].isin(targets)
)
metric_window_targets = metric_df[mask_window_targets].copy()

# Merge thresholds into window rows
metric_exceeds_currency_checkout = metric_window_targets.merge(
    metric_thresholds_currency_checkout,
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# Keep rows where value > global_P90
metric_exceeds_currency_checkout = metric_exceeds_currency_checkout[
    metric_exceeds_currency_checkout['value'] > metric_exceeds_currency_checkout['global_P90']
].copy()

# Compute ratio = value / global_P95 (handle division by zero by leaving inf)
metric_exceeds_currency_checkout['ratio'] = metric_exceeds_currency_checkout['value'] / metric_exceeds_currency_checkout['global_P95']

# Format timestamp as ISO UTC string for output
metric_exceeds_currency_checkout['timestamp_iso'] = metric_exceeds_currency_checkout['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Select and order desired columns
cols_out = ['timestamp_iso', 'cmdb_id', 'kpi_name', 'value', 'global_P95', 'global_P90', 'ratio']
metric_exceeds_currency_checkout = metric_exceeds_currency_checkout[cols_out]

# 3) Return up to top 20 rows sorted by ratio desc (most severe first)
metric_exceeds_currency_checkout = metric_exceeds_currency_checkout.sort_values(by='ratio', ascending=False).reset_index(drop=True)
metric_exceeds_currency_checkout_head = metric_exceeds_currency_checkout.head(20)

# Ensure full threshold and full exceeds DataFrames are stored in kernel variables as requested:
# metric_thresholds_currency_checkout (already set) and metric_exceeds_currency_checkout (set above).
metric_thresholds_currency_checkout, metric_exceeds_currency_checkout_head

```
Out[6]:
```
Summary of metric exceedances (currencyservice and checkoutservice) during the incident window (2024-01-17 03:53:39–04:23:39 UTC):

Key findings (top severity by ratio value / global_P95):
- checkoutservice — workload at 2024-01-17T03:58:00Z: ratio ≈ 1.195 (value 1.2556 vs global_P95 1.0507) — largest relative spike.
- checkoutservice — latency-90 at 2024-01-17T04:18:00Z: ratio ≈ 1.156 (value 2.6655 vs global_P95 2.3061).
- checkoutservice — latency-50 at 2024-01-17T04:19:00Z: ratio ≈ 1.132 (value 0.8313 vs global_P95 0.7344).
- currencyservice — workload at 2024-01-17T03:58:00Z: ratio ≈ 1.073 (value 87.5909 vs global_P95 81.6341).
- currencyservice — several smaller exceedances (latency-50, latency-90, mem, cpu, socket) with ratios mostly in 1.01–1.05 range (moderate deviations).

Observations and interpretation:
- Checkoutservice shows the most severe relative metric spikes (highest ratios), notably workload and later latency spikes around 04:18–04:19 UTC.
- Currencyservice had elevated workload and some metric exceedances (including memory and latency) early in the window (~03:58 and later), but their magnitude relative to global P95 is smaller than the top checkoutservice spikes.
- Global P95/P90 thresholds for each (cmdb_id, kpi_name) were computed over the full series and are stored in metric_thresholds_currency_checkout. Full exceed rows are in metric_exceeds_currency_checkout.

Conclusion:
- Both services show metric-level anomalies during the window: checkoutservice shows stronger/severe spikes (workload and latency), while currencyservice shows elevated load and modest metric exceedances. This supports investigating both services (checkoutservice for high-latency impact; currencyservice for increased load), but checkoutservice contains the largest relative metric anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id    kpi_name  count    global_P95    global_P90
0   checkoutservice         cpu     25  4.649672e-01  4.569296e-01
1   checkoutservice  latency-50     25  7.344464e-01  6.724038e-01
2   checkoutservice  latency-90     25  2.306130e+00  2.103525e+00
3   checkoutservice         mem     25  1.184507e+07  1.182738e+07
4   checkoutservice      socket     25  9.000000e+00  9.000000e+00
5   checkoutservice    workload     25  1.050717e+00  1.019090e+00
6   currencyservice         cpu     25  3.958943e+01  3.940357e+01
7   currencyservice       error     25  0.000000e+00  0.000000e+00
8   currencyservice  latency-50     25  1.043986e-01  1.025262e-01
9   currencyservice  latency-90     25  7.544970e-01  6.912868e-01
10  currencyservice         mem     25  2.516954e+08  2.386676e+08
11  currencyservice      socket     25  6.290000e+00  6.150000e+00
12  currencyservice    workload     25  8.163412e+01  8.038108e+01,            timestamp_iso          cmdb_id    kpi_name         value    global_P95    global_P90     ratio
0   2024-01-17T03:58:00Z  checkoutservice    workload  1.255550e+00  1.050717e+00  1.019090e+00  1.194946
1   2024-01-17T04:18:00Z  checkoutservice  latency-90  2.665540e+00  2.306130e+00  2.103525e+00  1.155850
2   2024-01-17T04:19:00Z  checkoutservice  latency-50  8.312864e-01  7.344464e-01  6.724038e-01  1.131854
3   2024-01-17T03:58:00Z  currencyservice    workload  8.759092e+01  8.163412e+01  8.038108e+01  1.072969
4   2024-01-17T04:20:00Z  currencyservice  latency-50  1.099837e-01  1.043986e-01  1.025262e-01  1.053498
5   2024-01-17T04:20:00Z  currencyservice  latency-90  7.910584e-01  7.544970e-01  6.912868e-01  1.048458
6   2024-01-17T04:11:00Z  currencyservice         mem  2.614594e+08  2.516954e+08  2.386676e+08  1.038793
7   2024-01-17T04:19:00Z  checkoutservice  latency-90  2.354500e+00  2.306130e+00  2.103525e+00  1.020974
8   2024-01-17T03:58:00Z  checkoutservice         cpu  4.740316e-01  4.649672e-01  4.569296e-01  1.019495
9   2024-01-17T04:10:00Z  currencyservice  latency-90  7.660173e-01  7.544970e-01  6.912868e-01  1.015269
10  2024-01-17T04:16:00Z  currencyservice      socket  6.366667e+00  6.290000e+00  6.150000e+00  1.012189
11  2024-01-17T04:19:00Z  currencyservice         mem  2.542910e+08  2.516954e+08  2.386676e+08  1.010312
12  2024-01-17T04:11:00Z  currencyservice         cpu  3.997479e+01  3.958943e+01  3.940357e+01  1.009734
13  2024-01-17T04:02:00Z  checkoutservice    workload  1.056750e+00  1.050717e+00  1.019090e+00  1.005742
14  2024-01-17T04:18:00Z  checkoutservice  latency-50  7.385899e-01  7.344464e-01  6.724038e-01  1.005642
15  2024-01-17T04:05:00Z  currencyservice    workload  8.189350e+01  8.163412e+01  8.038108e+01  1.003177
16  2024-01-17T04:17:00Z  checkoutservice         cpu  4.663831e-01  4.649672e-01  4.569296e-01  1.003045
17  2024-01-17T04:20:00Z  currencyservice      socket  6.300000e+00  6.290000e+00  6.150000e+00  1.001590
18  2024-01-17T04:20:00Z  checkoutservice         mem  1.186150e+07  1.184507e+07  1.182738e+07  1.001387
19  2024-01-17T04:16:00Z  currencyservice  latency-50  1.044846e-01  1.043986e-01  1.025262e-01  1.000824)```
```

In [ ]:
# In[7]:

import pandas as pd

# reuse existing kernel variables: metric_df, metric_thresholds_currency_checkout, trace_agg_df
# incident window (UTC)
start = pd.to_datetime("2024-01-17 03:53:39", utc=True)
end = pd.to_datetime("2024-01-17 04:23:39", utc=True)
targets = ['currencyservice', 'checkoutservice']

# Ensure thresholds exist; if not, compute from full series
if 'metric_thresholds_currency_checkout' not in globals() or metric_thresholds_currency_checkout is None or metric_thresholds_currency_checkout.shape[0] == 0:
    metric_thresholds_currency_checkout = (
        metric_df[metric_df['cmdb_id'].isin(targets)]
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(
            count='size',
            global_P95=lambda s: s.quantile(0.95),
            global_P90=lambda s: s.quantile(0.90)
        )
        .reset_index()
    )

# Prepare windowed metric data for targets
metric_window = metric_df[
    (metric_df['timestamp'] >= start) &
    (metric_df['timestamp'] <= end) &
    (metric_df['cmdb_id'].isin(targets))
].copy()

# Merge thresholds into window rows to evaluate exceedances
mw = metric_window.merge(
    metric_thresholds_currency_checkout[['cmdb_id', 'kpi_name', 'global_P95', 'global_P90']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# Create exceed flags
mw['exceed_P95'] = mw['value'] > mw['global_P95']
mw['exceed_P90'] = mw['value'] > mw['global_P90']

# Compute earliest exceed timestamps and max value per (cmdb_id, kpi_name) within window
# Earliest P95
earliest_p95 = (
    mw[mw['exceed_P95']]
    .groupby(['cmdb_id', 'kpi_name'])['timestamp']
    .min()
    .reset_index()
    .rename(columns={'timestamp': 'earliest_P95_exceed_dt'})
)
# Earliest P90
earliest_p90 = (
    mw[mw['exceed_P90']]
    .groupby(['cmdb_id', 'kpi_name'])['timestamp']
    .min()
    .reset_index()
    .rename(columns={'timestamp': 'earliest_P90_exceed_dt'})
)
# Max value in window per KPI
max_in_window = (
    mw.groupby(['cmdb_id', 'kpi_name'])['value']
    .max()
    .reset_index()
    .rename(columns={'value': 'max_value_in_window'})
)

# Start from thresholds to ensure all KPIs are present
earliest_metric_exceeds_cmp = metric_thresholds_currency_checkout.merge(
    max_in_window, on=['cmdb_id', 'kpi_name'], how='left'
).merge(
    earliest_p95, on=['cmdb_id', 'kpi_name'], how='left'
).merge(
    earliest_p90, on=['cmdb_id', 'kpi_name'], how='left'
)

# Compute ratio_at_max (handle missing global_P95 or max value)
earliest_metric_exceeds_cmp['ratio_at_max'] = earliest_metric_exceeds_cmp['max_value_in_window'] / earliest_metric_exceeds_cmp['global_P95']

# Format earliest timestamps as ISO UTC strings (or NaN)
for col_dt, col_out in [('earliest_P95_exceed_dt', 'earliest_P95_exceed'), ('earliest_P90_exceed_dt', 'earliest_P90_exceed')]:
    if col_dt in earliest_metric_exceeds_cmp.columns:
        earliest_metric_exceeds_cmp[col_out] = pd.to_datetime(earliest_metric_exceeds_cmp[col_dt], utc=True, errors='coerce').dt.strftime('%Y-%m-%dT%H:%M:%SZ')
        # keep NaN for missing
        earliest_metric_exceeds_cmp[col_out] = earliest_metric_exceeds_cmp[col_out].where(earliest_metric_exceeds_cmp[col_dt].notna(), pd.NA)
        earliest_metric_exceeds_cmp.drop(columns=[col_dt], inplace=True)
    else:
        earliest_metric_exceeds_cmp[col_out] = pd.NA

# Reorder columns as requested
earliest_metric_exceeds_cmp = earliest_metric_exceeds_cmp[[
    'cmdb_id', 'kpi_name', 'earliest_P95_exceed', 'earliest_P90_exceed',
    'max_value_in_window', 'global_P95', 'global_P90', 'ratio_at_max'
]]

# 2) Per-service summary: earliest P95 and P90 across all KPIs
# Convert earliest strings back to datetime for min calculation (coerce NA)
ep = earliest_metric_exceeds_cmp.copy()
ep['earliest_P95_dt'] = pd.to_datetime(ep['earliest_P95_exceed'], utc=True, errors='coerce')
ep['earliest_P90_dt'] = pd.to_datetime(ep['earliest_P90_exceed'], utc=True, errors='coerce')

earliest_metric_summary_cmp = ep.groupby('cmdb_id').agg(
    service_earliest_P95_dt=('earliest_P95_dt', 'min'),
    service_earliest_P90_dt=('earliest_P90_dt', 'min')
).reset_index()

# Format service-level timestamps as ISO strings or NA
earliest_metric_summary_cmp['service_earliest_P95'] = earliest_metric_summary_cmp['service_earliest_P95_dt'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
earliest_metric_summary_cmp['service_earliest_P95'] = earliest_metric_summary_cmp['service_earliest_P95'].where(earliest_metric_summary_cmp['service_earliest_P95_dt'].notna(), pd.NA)
earliest_metric_summary_cmp['service_earliest_P90'] = earliest_metric_summary_cmp['service_earliest_P90_dt'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
earliest_metric_summary_cmp['service_earliest_P90'] = earliest_metric_summary_cmp['service_earliest_P90'].where(earliest_metric_summary_cmp['service_earliest_P90_dt'].notna(), pd.NA)

earliest_metric_summary_cmp = earliest_metric_summary_cmp[['cmdb_id', 'service_earliest_P95', 'service_earliest_P90']]

# 3) Trace anomalies: filter trace_agg_df to entries where trace_name mentions currencyservice or checkoutservice
# and earliest_timestamp_above_p95 is within the incident window
t = trace_agg_df.copy()

# ensure earliest_timestamp_above_p95 exists and is parsed
if 'earliest_timestamp_above_p95' in t.columns:
    t['_earliest_dt'] = pd.to_datetime(t['earliest_timestamp_above_p95'], utc=True, errors='coerce')
else:
    t['_earliest_dt'] = pd.NaT

mask_name = t['trace_name'].str.contains('currencyservice|checkoutservice', regex=True, na=False)
mask_time = (t['_earliest_dt'] >= start) & (t['_earliest_dt'] <= end)
trace_anom_cmp = t[mask_name & mask_time].copy()

# select requested columns and limit to top 50 rows combined (we'll cap this table to 50)
trace_anom_cmp = trace_anom_cmp[[
    'cmdb_id', 'trace_name', 'earliest_timestamp_above_p95', 'count_above_p95', 'global_P95', 'max_value_in_window'
]].reset_index(drop=True)

# Limit results to keep combined outputs compact. We'll cap each table but ensure total not excessive.
earliest_metric_exceeds_cmp = earliest_metric_exceeds_cmp.sort_values(by=['cmdb_id','kpi_name']).reset_index(drop=True).head(50)
earliest_metric_summary_cmp = earliest_metric_summary_cmp.sort_values('cmdb_id').reset_index(drop=True).head(50)
trace_anom_cmp = trace_anom_cmp.sort_values(by='earliest_timestamp_above_p95').reset_index(drop=True).head(50)

# Store variables in kernel as requested
# earliest_metric_exceeds_cmp, earliest_metric_summary_cmp, trace_anom_cmp

# Return the three compact tables (variables) — IPython will display them
earliest_metric_exceeds_cmp, earliest_metric_summary_cmp, trace_anom_cmp

```
Out[7]:
```
```python
summary = (
    "Findings (plain English):\n\n"
    "1) Timeline & earliest metric exceedances:\n"
    "- Both services show their earliest P90/P95-level metric exceedances at ~2024-01-17T03:58:00Z (service-level earliest P95/P90 for checkoutservice and currencyservice = 2024-01-17T03:58:00Z).\n"
    "- Checkoutservice: earliest P95 hits at 03:58 for workload and CPU; stronger latency P95 exceedances occur later at ~04:18–04:19 UTC.\n"
    "- Currencyservice: earliest P95 hit for workload at 03:58; memory and CPU P95 exceedances appear later (~04:11–04:16 UTC).\n\n"
    "2) Severity (metrics):\n"
    "- The largest relative spike (value / global_P95) is checkoutservice.workload at 03:58 (ratio ≈ 1.19). Checkoutservice also shows the largest latency ratios at ~04:18.\n"
    "- Currencyservice shows notable workload increase at 03:58 (ratio ≈ 1.07) and modest exceedances in mem/latency later (ratios ~1.01–1.05).\n\n"
    "3) Trace evidence:\n"
    "- Trace anomalies show heavy traffic between checkoutservice and currencyservice starting ~03:56–03:58 (row_count spikes), and a concentrated set of checkoutservice trace latency anomalies around ~04:14–04:18.\n\n"
    "Conclusion and recommendation:\n"
    "- Currencyservice experienced an early traffic surge and moderate metric exceedances at ~03:58, making it a likely contributor to early load. However, checkoutservice exhibits both the largest relative workload spike early and significant latency spikes later (~04:18).\n"
    "- This telemetry does not conclusively prove currencyservice as the sole root cause. Investigate: (a) the traffic surge at ~03:56–03:58 between checkout and currency; (b) currencyservice resource use around 03:56–04:12; and (c) checkoutservice request/latency traces and dependencies around 04:14–04:19 to determine causal chain."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id    kpi_name   earliest_P95_exceed   earliest_P90_exceed  max_value_in_window    global_P95    global_P90  ratio_at_max
0   checkoutservice         cpu  2024-01-17T03:58:00Z  2024-01-17T03:58:00Z         4.740316e-01  4.649672e-01  4.569296e-01      1.019495
1   checkoutservice  latency-50  2024-01-17T04:18:00Z  2024-01-17T04:10:00Z         8.312864e-01  7.344464e-01  6.724038e-01      1.131854
2   checkoutservice  latency-90  2024-01-17T04:18:00Z  2024-01-17T04:10:00Z         2.665540e+00  2.306130e+00  2.103525e+00      1.155850
3   checkoutservice         mem  2024-01-17T04:00:00Z  2024-01-17T04:00:00Z         1.186150e+07  1.184507e+07  1.182738e+07      1.001387
4   checkoutservice      socket                  <NA>                  <NA>         9.000000e+00  9.000000e+00  9.000000e+00      1.000000
5   checkoutservice    workload  2024-01-17T03:58:00Z  2024-01-17T03:58:00Z         1.255550e+00  1.050717e+00  1.019090e+00      1.194946
6   currencyservice         cpu  2024-01-17T04:11:00Z  2024-01-17T04:10:00Z         3.997479e+01  3.958943e+01  3.940357e+01      1.009734
7   currencyservice       error                  <NA>                  <NA>         0.000000e+00  0.000000e+00  0.000000e+00           NaN
8   currencyservice  latency-50  2024-01-17T04:16:00Z  2024-01-17T04:10:00Z         1.099837e-01  1.043986e-01  1.025262e-01      1.053498
9   currencyservice  latency-90  2024-01-17T04:10:00Z  2024-01-17T04:10:00Z         7.910584e-01  7.544970e-01  6.912868e-01      1.048458
10  currencyservice         mem  2024-01-17T04:11:00Z  2024-01-17T04:11:00Z         2.614594e+08  2.516954e+08  2.386676e+08      1.038793
11  currencyservice      socket  2024-01-17T04:16:00Z  2024-01-17T04:11:00Z         6.366667e+00  6.290000e+00  6.150000e+00      1.012189
12  currencyservice    workload  2024-01-17T03:58:00Z  2024-01-17T03:58:00Z         8.759092e+01  8.163412e+01  8.038108e+01      1.072969,            cmdb_id  service_earliest_P95  service_earliest_P90
0  checkoutservice  2024-01-17T03:58:00Z  2024-01-17T03:58:00Z
1  currencyservice  2024-01-17T03:58:00Z  2024-01-17T03:58:00Z,                   cmdb_id                                trace_name earliest_timestamp_above_p95  count_above_p95  global_P95  max_value_in_window
0   productcatalogservice   trace.from_checkoutservice.duration_p95         2024-01-17T03:56:00Z                2    0.000039             0.000085
1            emailservice  trace.from_checkoutservice.duration_mean         2024-01-17T03:57:00Z                2    0.000372             0.000384
2   productcatalogservice      trace.from_checkoutservice.row_count         2024-01-17T03:58:00Z                1   75.000000            95.000000
3         checkoutservice      trace.from_checkoutservice.row_count         2024-01-17T03:58:00Z                2  282.200000           322.000000
4                    root        trace.to_checkoutservice.row_count         2024-01-17T03:58:00Z                2  132.200000           157.000000
..                    ...                                       ...                          ...              ...         ...                  ...
27         paymentservice   trace.from_checkoutservice.duration_p95         2024-01-17T04:18:00Z                2    0.000612             0.001030
28        checkoutservice     trace.to_checkoutservice.duration_p95         2024-01-17T04:18:00Z                2    0.008538             0.042266
29                   root    trace.to_checkoutservice.duration_mean         2024-01-17T04:18:00Z                2    0.183627             0.254955
30        checkoutservice   trace.from_checkoutservice.duration_p95         2024-01-17T04:18:00Z                2    0.008538             0.042266
31        frontendservice    trace.to_checkoutservice.duration_mean         2024-01-17T04:18:00Z                2    0.959058             1.419229

[32 rows x 6 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```